In [1]:
import random

from pathlib import Path
import pandas as pd
from tqdm import tqdm
import numpy as np

import utils

In [2]:
folder_path = Path("../results/trees/mixed/")
result_folder = Path("../results/processed/")
result_folder.mkdir(parents=True, exist_ok=True)

output_fname = Path("coalescence_sackin_mixed.pkl")
output_fpath = result_folder / output_fname

#### Process simulation trees

In [3]:
iterations = 200 # Number of coalescence calculations per sample
checkpoint_interval = 60
num_samples = 10 # Number of samples per file

# Load checkpoint file 
temp_file = output_fname.stem + ".tmp"
temp_file_path = result_folder / temp_file

# if temp_file_path.exists():
#     checkpoint_df = pd.read_pickle(temp_file_path)
#     data = checkpoint_df.to_dict(orient='list')
#     completed_combinations = set(zip(data['graph_type'], data['rep']))
#     print("Resuming from checkpoint.")

# # Create new df 
# else:
data = {'graph_type': [], 
        'graph_name': [],
        'rep': [],
        'mean_coalescence_time': [], 
        'variance_coalescence_time': [],
        'sackin': [], 
        'sackin_variance': [],
        'Js': [], 
        'variance_Js': []
        }
completed_combinations = set()

In [4]:
count = 0
for file_path in tqdm(list(folder_path.glob('*_s0.1.txt'))):
    graph_name = file_path.stem
    graph_type = graph_name.split('_')[0]
    rep_idx = int(graph_name.split('_')[3])
    
    if (graph_type, rep_idx) in completed_combinations:
        continue
    else:
        tree_path = folder_path / Path("{0}_tree.txt".format(graph_name))
        final_path = folder_path / Path("{0}_list.txt".format(graph_name))
        
        if not tree_path.exists() or not final_path.exists():
            print(f"Skipping missing paths: {tree_path}, {final_path}")
            continue
            
        # Load the tree files
        tree_samples = utils.count_non_empty_lines(tree_path)
        final_samples = utils.count_non_empty_lines(final_path)
        num_samples = min(tree_samples, final_samples, num_samples)
        if num_samples == 0:
            print(f"Skipping empty files: {tree_path}, {final_path}")
            continue
            
        sample_coalescence_times = []
        sackinL = []
        Js = []
        for sample_idx in range(num_samples):
            tree_dict = utils.build_tree(tree_path, sample_idx)
            tree = tree_dict[0]
            
            # Calculate J index
            utils.compute_subtree_sizes(tree)
            for node in tree.traverse():
                utils.compute_balance_scores(node)
            J1 = utils.compute_normalized_balance_index(tree)
            Js.append(J1)
                
            distribution = utils.get_final_mutations_L(final_path, sample_idx)
            path = utils.find_all_nodes(tree, set(distribution))
            tree.prune(path)
            
            # Calculate the coalescence time
            coalescence_times = []
            for _ in range(iterations):
                try:
                    coalescence_times.append(
                        utils.get_coalescence_time(
                            tree, random.sample(distribution, 2)
                        )
                    )
                except:
                    pass
            sample_avg_time = np.mean(coalescence_times)
            sample_coalescence_times.append(sample_avg_time)

            # Calculate Sackin index
            sackin_index = utils.get_sackin_index(tree)
            sackinL.append(sackin_index)
            
            
        # Calculate mean and variance across all samples  
        mean_coalescence_time = np.mean(np.mean(sample_coalescence_times))
        variance_coalescence_time = np.mean(np.var(sample_coalescence_times))
        
        data['graph_type'].append(graph_type)
        data['graph_name'].append(graph_name.split('_mu')[0])
        data['rep'].append(rep_idx)
        data['mean_coalescence_time'].append(mean_coalescence_time)
        data['variance_coalescence_time'].append(variance_coalescence_time)
        data['sackin'].append(np.mean(sackinL))
        data['sackin_variance'].append(np.var(sackinL))
        data['Js'].append(np.mean(np.mean(Js)))
        data['variance_Js'].append(np.var(Js))
            
        count += 1

        if count % checkpoint_interval == 0:
            checkpoint_df = pd.DataFrame(data)
            checkpoint_df.to_pickle(temp_file_path)
            print(f"Checkpoint saved at iteration {count}")

 50%|█████     | 60/120 [00:16<00:15,  3.90it/s]

Checkpoint saved at iteration 60


100%|██████████| 120/120 [00:31<00:00,  3.79it/s]

Checkpoint saved at iteration 120


In [5]:
df = pd.DataFrame(data)
df = df.sort_values(['graph_type', 'rep']).reset_index(drop=True)
df.to_pickle(output_fpath)
print(f"Processed results saved at {output_fname}.")

Processed results saved at coalescence_sackin_mixed.pkl.
